# Swarm-Augmented Time Series Forecasting (SATF)
## Combining MiroFish-Style Swarm Intelligence with Google TimesFM

**A novel approach that bridges qualitative collective intelligence with quantitative foundation model forecasting.**

### The Core Idea

Two powerful prediction paradigms exist in isolation:

1. **MiroFish** — A swarm intelligence engine that spawns hundreds of AI agents with unique personalities to simulate social dynamics and predict outcomes through emergent collective behavior.
2. **Google TimesFM** — A decoder-only foundation model pre-trained on 400B+ real-world time points for zero-shot time series forecasting.

**Nobody has combined them.** MiroFish produces qualitative sentiment trajectories. TimesFM produces quantitative point/quantile forecasts. This notebook bridges the gap:

- We use **Claude to simulate MiroFish-style multi-agent swarm debates**, extracting numerical sentiment scores at each round
- We feed those **sentiment trajectories as signal modifiers** into TimesFM's quantitative forecasts
- Claude then serves as the **meta-reasoning layer**, synthesizing both signals into unified predictions with narrative explanations

### Table of Contents

1. [Setup and Installation](#setup)
2. [Define the Prediction Scenario](#scenario)
3. [Swarm Simulation — Multi-Agent Debate via Claude](#swarm)
4. [Sentiment Trajectory Extraction](#sentiment)
5. [Baseline TimesFM Forecast](#baseline)
6. [Swarm-Augmented Forecast](#augmented)
7. [Claude Meta-Synthesis — Unified Prediction](#synthesis)
8. [Visualization and Analysis](#visualization)

<a id="setup"></a>
## 1. Setup and Installation

We need:
- **`anthropic`** — Claude API for swarm simulation and meta-synthesis
- **`timesfm`** — Google's time series foundation model
- **`numpy`** / **`matplotlib`** — Numerical computation and visualization

In [ ]:
%pip install anthropic timesfm[torch] matplotlib numpy

In [ ]:
import json
import re

import anthropic
import matplotlib.pyplot as plt
import numpy as np

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-6"

<a id="scenario"></a>
## 2. Define the Prediction Scenario

We'll forecast **U.S. electric vehicle adoption rate** over the next 12 months. This is a great test case because:
- It has both **quantitative signals** (historical sales data, market penetration %)
- And **qualitative drivers** (policy changes, public sentiment, tech breakthroughs, competitor moves)

We'll use synthetic but realistic historical data, then apply our swarm-augmented approach.

In [ ]:
# Scenario definition — this "reality seed" drives the swarm simulation
SCENARIO = {
    "topic": "U.S. Electric Vehicle Market Penetration Rate",
    "context": """
    Current state (April 2026):
    - EV market share in the U.S. is approximately 12.5% of new car sales
    - Tesla's market share has declined from 55% to 43% of the EV segment
    - Chinese EV manufacturers (BYD, NIO) are pushing to enter the U.S. market
    - The federal $7,500 EV tax credit remains in place but faces political pressure
    - Battery costs have dropped to ~$100/kWh, approaching the $80/kWh tipping point
    - Charging infrastructure has grown 40% YoY but still has rural coverage gaps
    - Several major automakers have delayed their EV-only transition timelines
    - Consumer sentiment is mixed: range anxiety declining, but affordability concerns rising
    """,
    "prediction_question": "What will the U.S. EV market penetration rate be over the next 12 months?",
    "forecast_horizon": 12,  # months
}

# Historical monthly EV market penetration % (synthetic but realistic, 36 months of history)
# Represents a gradual S-curve adoption with seasonal variation and noise
np.random.seed(42)
months = np.arange(36)
trend = 5.0 + 7.5 * (1 / (1 + np.exp(-0.15 * (months - 18))))  # S-curve from ~5% to ~12.5%
seasonal = 0.4 * np.sin(2 * np.pi * months / 12)  # Seasonal pattern
noise = np.random.normal(0, 0.2, len(months))
historical_data = trend + seasonal + noise

print(f"Historical data: {len(historical_data)} months")
print(f"Latest value: {historical_data[-1]:.1f}% market penetration")
print(f"Range: {historical_data.min():.1f}% — {historical_data.max():.1f}%")

# Plot historical data
plt.figure(figsize=(10, 4))
plt.plot(months, historical_data, "b-o", markersize=3, label="Historical EV Market Share %")
plt.xlabel("Month")
plt.ylabel("Market Penetration (%)")
plt.title("U.S. EV Market Penetration — Historical Data")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<a id="swarm"></a>
## 3. Swarm Simulation — Multi-Agent Debate via Claude

This is the MiroFish-inspired core. Instead of deploying the full MiroFish infrastructure (Node.js frontend, OASIS engine, Zep Cloud memory), we use **Claude to simulate the swarm dynamics directly**.

### How it works:
1. **Define diverse agent personas** — each with a unique background, bias, and reasoning style
2. **Run multi-round debates** — agents react to each other's arguments, shifting positions
3. **Extract structured scores** — each agent provides a numerical sentiment/confidence score per round
4. **Emergent behavior** — collective opinion evolves through social influence, not just averaging

This mirrors MiroFish's GraphRAG → Agent Generation → OASIS Simulation pipeline, but uses Claude as the simulation substrate.

In [ ]:
# Define the swarm: diverse agent personas inspired by MiroFish's agent generation
SWARM_AGENTS = [
    {
        "name": "Dr. Sarah Chen",
        "role": "Automotive Industry Analyst",
        "bias": "data-driven moderate",
        "background": "20 years covering the auto industry for a major investment bank. "
        "Focuses on supply chain economics and manufacturing capacity.",
    },
    {
        "name": "Marcus Rodriguez",
        "role": "EV Startup Founder",
        "bias": "tech-optimist bull",
        "background": "Founded an EV charging network startup. Believes passionately in the "
        "EV transition but understands the infrastructure challenges firsthand.",
    },
    {
        "name": "Janet O'Brien",
        "role": "Consumer Behavior Researcher",
        "bias": "skeptical realist",
        "background": "Professor studying technology adoption curves. Has published work "
        "showing that EV adoption lags behind media hype due to practical barriers.",
    },
    {
        "name": "Raj Patel",
        "role": "Energy Policy Advisor",
        "bias": "policy-focused pragmatist",
        "background": "Former DOE advisor who helped design the EV tax credit. "
        "Deeply understands how policy uncertainty affects market behavior.",
    },
    {
        "name": "Yuki Tanaka",
        "role": "Battery Technology Researcher",
        "bias": "technology-focused optimist",
        "background": "Leads a battery materials lab. Has insider knowledge on "
        "next-gen solid-state batteries and cost reduction trajectories.",
    },
    {
        "name": "Tom Brewer",
        "role": "Rural Auto Dealer",
        "bias": "ground-level skeptic",
        "background": "Runs three dealerships in the Midwest. Sees firsthand how "
        "customers react to EVs vs. trucks and hybrids in rural America.",
    },
    {
        "name": "Aisha Williams",
        "role": "Climate Finance Investor",
        "bias": "long-term structural bull",
        "background": "Manages a $2B climate-focused fund. Views EV adoption "
        "through the lens of capital flows and regulatory inevitability.",
    },
    {
        "name": "David Kowalski",
        "role": "Geopolitical Risk Analyst",
        "bias": "risk-aware contrarian",
        "background": "Tracks how tariffs, trade wars, and supply chain "
        "disruptions from China affect the EV market dynamics.",
    },
]

print(f"Swarm size: {len(SWARM_AGENTS)} agents")
for agent in SWARM_AGENTS:
    print(f"  • {agent['name']} — {agent['role']} ({agent['bias']})")

### Run the Multi-Round Swarm Debate

Each round, all agents respond to the evolving debate. Like MiroFish's OASIS simulation, agents can shift positions based on arguments from others — creating emergent collective intelligence rather than static polling.

In [ ]:
def run_swarm_round(agents, scenario, round_num, previous_debate):
    """Run one round of the swarm debate, collecting each agent's response."""
    round_results = []

    # Build the debate context from previous rounds
    debate_history = ""
    if previous_debate:
        debate_history = "\n\n--- PREVIOUS ROUND ARGUMENTS ---\n"
        for entry in previous_debate[-len(agents) :]:  # Show last round's arguments
            debate_history += f"\n**{entry['name']}** ({entry['role']}): {entry['argument']}\n"
            debate_history += f"  Sentiment: {entry['sentiment']}/10 | "
            debate_history += f"Predicted change: {entry['predicted_direction']}\n"

    for agent in agents:
        prompt = f"""You are {agent["name"]}, a {agent["role"]}.

Background: {agent["background"]}
Your natural analytical bias: {agent["bias"]}

SCENARIO:
Topic: {scenario["topic"]}
{scenario["context"]}

Question: {scenario["prediction_question"]}

This is Round {round_num} of a multi-expert debate.
{debate_history}

Based on your expertise and the arguments you've heard from others, provide:

1. Your argument (2-3 sentences, be specific and cite reasoning)
2. Your SENTIMENT score (1-10 scale):
   - 1-3: Very bearish (expect significant decline or stagnation)
   - 4-5: Moderately bearish/cautious
   - 6-7: Moderately bullish/optimistic
   - 8-10: Very bullish (expect strong acceleration)
3. Your predicted direction for EV market share over 12 months: UP, FLAT, or DOWN
4. Your confidence in your prediction (0-100%)

You MUST respond in this exact JSON format:
{{
    "argument": "your argument here",
    "sentiment": <number 1-10>,
    "predicted_direction": "UP" or "FLAT" or "DOWN",
    "confidence": <number 0-100>
}}

If other agents made compelling points, you may shift your position. Be authentic to your persona."""

        response = client.messages.create(
            model=MODEL,
            max_tokens=500,
            temperature=0.8,  # Higher temperature for diverse agent personalities
            messages=[{"role": "user", "content": prompt}],
        )

        response_text = response.content[0].text.strip()

        # Parse JSON from the response — handle markdown code blocks
        json_match = re.search(r"\{[^{}]*\}", response_text, re.DOTALL)
        if json_match:
            parsed = json.loads(json_match.group())
            parsed["name"] = agent["name"]
            parsed["role"] = agent["role"]
            parsed["round"] = round_num
            round_results.append(parsed)

    return round_results


# Run 5 rounds of debate (simulating MiroFish's temporal simulation steps)
NUM_ROUNDS = 5
all_debate_results = []

print(f"Running {NUM_ROUNDS}-round swarm debate with {len(SWARM_AGENTS)} agents...\n")

for round_num in range(1, NUM_ROUNDS + 1):
    print(f"--- Round {round_num} ---")
    round_results = run_swarm_round(SWARM_AGENTS, SCENARIO, round_num, all_debate_results)
    all_debate_results.extend(round_results)

    # Show round summary
    sentiments = [r["sentiment"] for r in round_results]
    avg_sentiment = np.mean(sentiments)
    directions = [r["predicted_direction"] for r in round_results]
    print(f"  Avg sentiment: {avg_sentiment:.1f}/10")
    print(f"  Directions: {', '.join(directions)}")
    print()

<a id="sentiment"></a>
## 4. Sentiment Trajectory Extraction

Now we extract the key innovation: **converting qualitative swarm debate into a quantitative time series** that can augment TimesFM's forecast.

We compute per-round metrics:
- **Swarm sentiment** — Weighted average sentiment (confidence-weighted)
- **Consensus strength** — How much agents agree (inverse of variance)
- **Directional momentum** — Net bullish vs. bearish signal

In [ ]:
def extract_sentiment_trajectory(debate_results, num_agents, num_rounds):
    """Convert swarm debate results into quantitative sentiment trajectories."""
    trajectory = {
        "rounds": [],
        "avg_sentiment": [],
        "weighted_sentiment": [],
        "consensus_strength": [],
        "bullish_ratio": [],
        "avg_confidence": [],
        "sentiment_momentum": [],  # Change in sentiment between rounds
    }

    for r in range(1, num_rounds + 1):
        round_data = [d for d in debate_results if d["round"] == r]
        if not round_data:
            continue

        sentiments = [d["sentiment"] for d in round_data]
        confidences = [d["confidence"] / 100.0 for d in round_data]
        directions = [d["predicted_direction"] for d in round_data]

        # Confidence-weighted sentiment
        weights = np.array(confidences)
        weighted_avg = np.average(sentiments, weights=weights)

        # Consensus: inverse of coefficient of variation (higher = more agreement)
        sentiment_std = np.std(sentiments)
        consensus = 1.0 / (1.0 + sentiment_std)

        # Bullish ratio
        bullish = sum(1 for d in directions if d == "UP") / len(directions)

        trajectory["rounds"].append(r)
        trajectory["avg_sentiment"].append(np.mean(sentiments))
        trajectory["weighted_sentiment"].append(weighted_avg)
        trajectory["consensus_strength"].append(consensus)
        trajectory["bullish_ratio"].append(bullish)
        trajectory["avg_confidence"].append(np.mean(confidences))

        # Momentum (change from previous round)
        if len(trajectory["avg_sentiment"]) > 1:
            momentum = trajectory["avg_sentiment"][-1] - trajectory["avg_sentiment"][-2]
        else:
            momentum = 0.0
        trajectory["sentiment_momentum"].append(momentum)

    return trajectory


sentiment_trajectory = extract_sentiment_trajectory(
    all_debate_results, len(SWARM_AGENTS), NUM_ROUNDS
)

# Display the trajectory
print("Swarm Sentiment Trajectory:")
print(
    f"{'Round':<8} {'Sentiment':<12} {'Weighted':<12} {'Consensus':<12} {'Bullish%':<12} {'Momentum':<12}"
)
print("-" * 68)
for i, r in enumerate(sentiment_trajectory["rounds"]):
    print(
        f"{r:<8} "
        f"{sentiment_trajectory['avg_sentiment'][i]:<12.2f}"
        f"{sentiment_trajectory['weighted_sentiment'][i]:<12.2f}"
        f"{sentiment_trajectory['consensus_strength'][i]:<12.2f}"
        f"{sentiment_trajectory['bullish_ratio'][i]:<12.1%}"
        f"{sentiment_trajectory['sentiment_momentum'][i]:<12.2f}"
    )

In [ ]:
# Visualize the swarm sentiment evolution across debate rounds
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Sentiment over rounds
axes[0, 0].plot(
    sentiment_trajectory["rounds"],
    sentiment_trajectory["avg_sentiment"],
    "ro-",
    label="Avg Sentiment",
)
axes[0, 0].plot(
    sentiment_trajectory["rounds"],
    sentiment_trajectory["weighted_sentiment"],
    "bs-",
    label="Confidence-Weighted",
)
axes[0, 0].axhline(y=5.5, color="gray", linestyle="--", alpha=0.5, label="Neutral")
axes[0, 0].set_xlabel("Debate Round")
axes[0, 0].set_ylabel("Sentiment (1-10)")
axes[0, 0].set_title("Swarm Sentiment Evolution")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Consensus strength
axes[0, 1].bar(
    sentiment_trajectory["rounds"],
    sentiment_trajectory["consensus_strength"],
    color="green",
    alpha=0.7,
)
axes[0, 1].set_xlabel("Debate Round")
axes[0, 1].set_ylabel("Consensus Strength")
axes[0, 1].set_title("Agent Consensus (Higher = More Agreement)")
axes[0, 1].grid(True, alpha=0.3)

# Bullish ratio
axes[1, 0].bar(
    sentiment_trajectory["rounds"],
    sentiment_trajectory["bullish_ratio"],
    color="blue",
    alpha=0.7,
)
axes[1, 0].axhline(y=0.5, color="gray", linestyle="--", alpha=0.5)
axes[1, 0].set_xlabel("Debate Round")
axes[1, 0].set_ylabel("Bullish Ratio")
axes[1, 0].set_title("% of Agents Predicting UP")
axes[1, 0].set_ylim(0, 1)
axes[1, 0].grid(True, alpha=0.3)

# Individual agent sentiment trajectories (the emergent behavior view)
for agent in SWARM_AGENTS:
    agent_data = [d for d in all_debate_results if d["name"] == agent["name"]]
    rounds = [d["round"] for d in agent_data]
    sents = [d["sentiment"] for d in agent_data]
    axes[1, 1].plot(rounds, sents, "o-", label=agent["name"], alpha=0.7, markersize=4)
axes[1, 1].set_xlabel("Debate Round")
axes[1, 1].set_ylabel("Sentiment (1-10)")
axes[1, 1].set_title("Individual Agent Trajectories")
axes[1, 1].legend(fontsize=6, loc="best")
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

<a id="baseline"></a>
## 5. Baseline TimesFM Forecast

Now we run Google TimesFM 2.5 on the raw historical data to get a **pure quantitative baseline** forecast. This is what you'd get without any swarm intelligence — just the numbers.

In [ ]:
import timesfm
import torch

torch.set_float32_matmul_precision("high")

# Load TimesFM 2.5 (200M parameter model with quantile prediction head)
tfm_model = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")

# Configure for monthly forecasting with probabilistic output
tfm_model.compile(
    timesfm.ForecastConfig(
        max_context=512,
        max_horizon=SCENARIO["forecast_horizon"],
        normalize_inputs=True,
        use_continuous_quantile_head=True,  # Enable probabilistic forecasts
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)

print("TimesFM 2.5 loaded successfully")

In [ ]:
# Run baseline forecast on historical data
baseline_point, baseline_quantiles = tfm_model.forecast(
    horizon=SCENARIO["forecast_horizon"],
    inputs=[historical_data],
)

baseline_forecast = baseline_point[0]  # Shape: (12,)
baseline_quantile_forecast = baseline_quantiles[0]  # Shape: (12, num_quantiles)

forecast_months = np.arange(
    len(historical_data), len(historical_data) + SCENARIO["forecast_horizon"]
)

print(f"Baseline forecast (next {SCENARIO['forecast_horizon']} months):")
for i, val in enumerate(baseline_forecast):
    print(f"  Month {i + 1}: {val:.2f}%")

# Plot baseline forecast
plt.figure(figsize=(12, 5))
plt.plot(months, historical_data, "b-o", markersize=3, label="Historical")
plt.plot(
    forecast_months, baseline_forecast, "r--o", markersize=3, label="TimesFM Baseline Forecast"
)

# Plot confidence bands from quantile forecasts
if baseline_quantile_forecast.ndim == 2 and baseline_quantile_forecast.shape[1] >= 2:
    lower = baseline_quantile_forecast[:, 0]  # Lower quantile
    upper = baseline_quantile_forecast[:, -1]  # Upper quantile
    plt.fill_between(
        forecast_months, lower, upper, alpha=0.2, color="red", label="Prediction Interval"
    )

plt.xlabel("Month")
plt.ylabel("Market Penetration (%)")
plt.title("TimesFM Baseline Forecast — Pure Quantitative")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<a id="augmented"></a>
## 6. Swarm-Augmented Forecast

**This is the novel combination.** We use the swarm's sentiment trajectory to modify TimesFM's baseline forecast:

### The SATF Algorithm:

1. **Sentiment Modifier** — The swarm's final consensus sentiment maps to a directional bias:
   - Sentiment > 5.5 → bullish modifier (scales forecast upward)
   - Sentiment < 5.5 → bearish modifier (scales forecast downward)
   - The magnitude depends on consensus strength (high agreement = stronger signal)

2. **Confidence Band Adjustment** — Low swarm consensus *widens* the prediction interval (more uncertainty), high consensus *tightens* it

3. **Momentum Overlay** — If swarm sentiment shifted strongly during debate, apply a trend acceleration/deceleration to the forecast curve

This bridges the qualitative-quantitative gap: **the swarm tells us WHERE the numbers might be wrong.**

In [ ]:
def swarm_augmented_forecast(baseline_forecast, baseline_quantiles, sentiment_trajectory):
    """Apply swarm intelligence signals to modify TimesFM's quantitative forecast."""
    horizon = len(baseline_forecast)

    # Extract final-round swarm signals
    final_sentiment = sentiment_trajectory["weighted_sentiment"][-1]
    final_consensus = sentiment_trajectory["consensus_strength"][-1]
    final_bullish_ratio = sentiment_trajectory["bullish_ratio"][-1]

    # Compute sentiment momentum (trend across all rounds)
    sentiments = sentiment_trajectory["weighted_sentiment"]
    if len(sentiments) >= 2:
        sentiment_trend = (sentiments[-1] - sentiments[0]) / len(sentiments)
    else:
        sentiment_trend = 0.0

    # --- 1. Directional Bias ---
    # Map sentiment (1-10) to a multiplier centered on 1.0
    # Sentiment of 5.5 = neutral (1.0x), 10 = 1.05x, 1 = 0.95x
    sentiment_deviation = (final_sentiment - 5.5) / 4.5  # Range: [-1, 1]
    # Scale by consensus: high agreement amplifies the signal
    max_adjustment = 0.05  # Maximum 5% adjustment to forecast
    directional_bias = sentiment_deviation * final_consensus * max_adjustment

    # Apply bias with increasing strength over the forecast horizon
    # (swarm insights matter more for longer-term predictions)
    time_weights = np.linspace(0.3, 1.0, horizon)
    augmented_forecast = baseline_forecast * (1 + directional_bias * time_weights)

    # --- 2. Momentum Overlay ---
    # If the swarm's sentiment shifted during debate, apply acceleration
    momentum_factor = sentiment_trend * 0.002  # Small but compounding
    momentum_curve = np.cumsum(np.full(horizon, momentum_factor))
    augmented_forecast = augmented_forecast + momentum_curve

    # --- 3. Confidence Band Adjustment ---
    augmented_quantiles = baseline_quantiles.copy()
    if augmented_quantiles.ndim == 2 and augmented_quantiles.shape[1] >= 2:
        # Low consensus = wider bands, high consensus = tighter bands
        band_scale = 1.0 + (1.0 - final_consensus) * 0.5  # 1.0 to 1.5x width
        mid = augmented_forecast
        for q in range(augmented_quantiles.shape[1]):
            augmented_quantiles[:, q] = (
                mid + (augmented_quantiles[:, q] - baseline_forecast) * band_scale
            )

    return (
        augmented_forecast,
        augmented_quantiles,
        {
            "directional_bias": directional_bias,
            "sentiment_trend": sentiment_trend,
            "final_sentiment": final_sentiment,
            "final_consensus": final_consensus,
            "final_bullish_ratio": final_bullish_ratio,
        },
    )


augmented_forecast, augmented_quantiles, swarm_signals = swarm_augmented_forecast(
    baseline_forecast, baseline_quantile_forecast, sentiment_trajectory
)

print("Swarm Augmentation Signals:")
print(f"  Final sentiment: {swarm_signals['final_sentiment']:.2f}/10")
print(f"  Consensus strength: {swarm_signals['final_consensus']:.3f}")
print(f"  Bullish ratio: {swarm_signals['final_bullish_ratio']:.1%}")
print(f"  Directional bias: {swarm_signals['directional_bias']:+.4f}")
print(f"  Sentiment trend: {swarm_signals['sentiment_trend']:+.3f}/round")
print()
print("Forecast Comparison:")
print(f"{'Month':<8} {'Baseline':<12} {'Augmented':<12} {'Difference':<12}")
print("-" * 44)
for i in range(len(baseline_forecast)):
    diff = augmented_forecast[i] - baseline_forecast[i]
    print(f"{i + 1:<8} {baseline_forecast[i]:<12.2f} {augmented_forecast[i]:<12.2f} {diff:<+12.3f}")

<a id="synthesis"></a>
## 7. Claude Meta-Synthesis — Unified Prediction

The final piece: Claude acts as the **meta-reasoning layer**, consuming both the quantitative forecast and the qualitative swarm debate to produce a unified narrative prediction.

This is something neither MiroFish nor TimesFM can do alone — bridging numbers and narrative into actionable insight.

In [ ]:
# Prepare the full context for Claude's meta-synthesis
# Collect final-round arguments from all agents
final_round_args = [d for d in all_debate_results if d["round"] == NUM_ROUNDS]
debate_summary = "\n".join(
    f"- {d['name']} ({d['role']}, sentiment {d['sentiment']}/10): {d['argument']}"
    for d in final_round_args
)

synthesis_prompt = f"""You are a meta-analyst synthesizing two independent prediction signals into a unified forecast.

## SIGNAL 1: Quantitative Forecast (Google TimesFM 2.5)
A time series foundation model trained on 400B+ data points analyzed 36 months of historical EV market penetration data.

Baseline forecast (next 12 months, monthly EV market share %):
{chr(10).join(f"  Month {i + 1}: {v:.2f}%" for i, v in enumerate(baseline_forecast))}

## SIGNAL 2: Swarm Intelligence (MiroFish-Style Multi-Agent Debate)
8 expert agents with diverse backgrounds debated for {NUM_ROUNDS} rounds.

Final sentiment trajectory:
- Starting swarm sentiment: {sentiment_trajectory["weighted_sentiment"][0]:.2f}/10
- Final swarm sentiment: {sentiment_trajectory["weighted_sentiment"][-1]:.2f}/10
- Consensus strength: {sentiment_trajectory["consensus_strength"][-1]:.3f}
- Bullish ratio: {sentiment_trajectory["bullish_ratio"][-1]:.1%}

Final round expert positions:
{debate_summary}

## SWARM-AUGMENTED FORECAST
After applying swarm signals to the baseline:
{chr(10).join(f"  Month {i + 1}: {v:.2f}% (delta: {v - baseline_forecast[i]:+.3f})" for i, v in enumerate(augmented_forecast))}

## YOUR TASK
Produce a meta-synthesis that:
1. Identifies where the quantitative model and the swarm AGREE (high-confidence zones)
2. Identifies where they DIVERGE (areas of maximum uncertainty)
3. Explains which qualitative factors the pure numbers miss
4. Provides your unified 12-month prediction with confidence levels
5. Flags the top 3 "wild card" events that could break the forecast

Be specific. Cite which agents' arguments were most compelling and why.
Reference specific months where the two signals agree or diverge."""

synthesis = client.messages.create(
    model=MODEL,
    max_tokens=2000,
    temperature=0.3,  # Lower temperature for analytical synthesis
    messages=[{"role": "user", "content": synthesis_prompt}],
)

print("=" * 80)
print("CLAUDE META-SYNTHESIS: Unified Prediction Report")
print("=" * 80)
print(synthesis.content[0].text)

<a id="visualization"></a>
## 8. Final Visualization — The Complete SATF Picture

Putting it all together: historical data, baseline TimesFM forecast, and the swarm-augmented forecast side by side.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={"height_ratios": [3, 1]})

# --- Top panel: Main forecast comparison ---
ax1 = axes[0]
ax1.plot(months, historical_data, "b-o", markersize=3, linewidth=1.5, label="Historical Data")
ax1.plot(
    forecast_months,
    baseline_forecast,
    "r--s",
    markersize=4,
    linewidth=1.5,
    label="TimesFM Baseline",
)
ax1.plot(
    forecast_months,
    augmented_forecast,
    "g-D",
    markersize=4,
    linewidth=2,
    label="Swarm-Augmented (SATF)",
)

# Baseline confidence band
if baseline_quantile_forecast.ndim == 2 and baseline_quantile_forecast.shape[1] >= 2:
    ax1.fill_between(
        forecast_months,
        baseline_quantile_forecast[:, 0],
        baseline_quantile_forecast[:, -1],
        alpha=0.1,
        color="red",
        label="Baseline CI",
    )

# Augmented confidence band
if augmented_quantiles.ndim == 2 and augmented_quantiles.shape[1] >= 2:
    ax1.fill_between(
        forecast_months,
        augmented_quantiles[:, 0],
        augmented_quantiles[:, -1],
        alpha=0.15,
        color="green",
        label="Swarm-Augmented CI",
    )

# Mark the transition point
ax1.axvline(x=len(historical_data) - 0.5, color="gray", linestyle=":", alpha=0.5)
ax1.text(
    len(historical_data) + 0.5,
    ax1.get_ylim()[1] * 0.95,
    "Forecast",
    fontsize=9,
    color="gray",
)

ax1.set_xlabel("Month")
ax1.set_ylabel("EV Market Penetration (%)")
ax1.set_title(
    "Swarm-Augmented Time Series Forecasting (SATF)\n"
    "MiroFish Swarm Intelligence + Google TimesFM 2.5 + Claude Meta-Synthesis",
    fontsize=13,
)
ax1.legend(loc="upper left")
ax1.grid(True, alpha=0.3)

# --- Bottom panel: Swarm signal overlay ---
ax2 = axes[1]

# Show sentiment trajectory alongside the forecast divergence
ax2_left = ax2
rounds_x = np.linspace(forecast_months[0], forecast_months[-1], len(sentiment_trajectory["rounds"]))
ax2_left.bar(
    rounds_x,
    sentiment_trajectory["weighted_sentiment"],
    width=0.6,
    color="purple",
    alpha=0.6,
    label="Swarm Sentiment",
)
ax2_left.axhline(y=5.5, color="gray", linestyle="--", alpha=0.5)
ax2_left.set_ylabel("Swarm Sentiment (1-10)", color="purple")
ax2_left.set_xlabel("Forecast Month")
ax2_left.set_ylim(0, 10)
ax2_left.tick_params(axis="y", labelcolor="purple")

# Overlay forecast divergence on right axis
ax2_right = ax2.twinx()
divergence = augmented_forecast - baseline_forecast
ax2_right.plot(
    forecast_months,
    divergence,
    "g-o",
    markersize=3,
    linewidth=1.5,
    label="Forecast Divergence",
)
ax2_right.axhline(y=0, color="gray", linestyle="-", alpha=0.3)
ax2_right.set_ylabel("SATF - Baseline (pp)", color="green")
ax2_right.tick_params(axis="y", labelcolor="green")

# Combined legend
lines1, labels1 = ax2_left.get_legend_handles_labels()
lines2, labels2 = ax2_right.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

## What Makes This Novel

**No one has combined these three systems before:**

| Component | Role | What It Provides |
|---|---|---|
| **MiroFish-style Swarm** (via Claude) | Qualitative intelligence | Emergent sentiment trajectories from diverse expert agents debating over rounds |
| **Google TimesFM 2.5** | Quantitative forecasting | Zero-shot point + quantile forecasts from a foundation model trained on 400B+ data points |
| **Claude Meta-Synthesis** | Reasoning layer | Reconciliation of where numbers and narrative agree/diverge, with explanation |

### The SATF Framework

**Swarm-Augmented Time Series Forecasting** bridges the qualitative-quantitative gap:

1. **Swarm signals modify forecast direction** — When experts collectively agree on a trend the numbers might miss (policy changes, tech breakthroughs), the forecast shifts accordingly
2. **Consensus modulates confidence bands** — Disagreement among experts *widens* prediction intervals; agreement *narrows* them
3. **Meta-synthesis explains the forecast** — Instead of a black-box number, you get a narrative explaining which factors drive the prediction

### Extensions

- **Real MiroFish integration** — Feed actual MiroFish simulation outputs (with 1000+ agents) as the swarm signal
- **Live data feeds** — Replace synthetic data with real market data via financial APIs
- **Multi-scenario analysis** — Run multiple swarm simulations with different "reality seeds" and compare forecast distributions
- **Feedback loops** — Use forecast accuracy to tune the swarm-to-quantitative weighting over time
- **Domain transfer** — Apply SATF to any domain where both quantitative history and qualitative expert opinion matter (crypto, policy impact, supply chain)